# 3B — Tests d'hétérogénéité et fusion complémentaire niveau Modèle

**Objectif** : évaluer l'hétérogénéité résiduelle entre les classes `CHR_apres_fusion` (issues de `fusion_glouton`) et décider des fusions complémentaires au niveau Modèle.

**Protocole méthodologique**
1. Calcul de l'OVL (Weitzman 1970) sur toutes les paires adjacentes
2. Calibration empirique du seuil OVL par stabilité bootstrap (ARI, Hubert & Arabie 1985)
3. Fusion transitive avec le seuil calibré
4. Diagnostic documentaire (batterie complète de tests)
5. Application à la base → `HLC_test_heterogeneite`

**Justification du critère** : l'OVL est non-paramétrique, indépendant de la taille d'échantillon, et puissant pour détecter les différences de *forme* de distribution (Komaba, Johno & Nakamoto 2023). Les tests classiques (Welch/KS/ANOVA) saturent à grand effectif (p≈0 systématique sur 20k+ obligors) et restent **documentaires**.

## 0. Imports et configuration

Le `reload` force la prise en compte des dernières modifications de `tests_heterogeneite.py`. La fonction `_default_suite` est importée explicitement car elle est préfixée par `_` (non exportée par `import *`).

In [ ]:
import importlib
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

import tests_heterogeneite as th
importlib.reload(th)

from tests_heterogeneite import (
    FusionComparator,
    OVLTest,
    HeterogeneityTestSuite,
)
from tests_heterogeneite import _default_suite

# --- colonnes du projet ---
CLASS_COL   = "CHR_apres_fusion"   # classes d'entrée (post fusion_glouton)
LGD_COL     = "lgd_obligor"        # LGD au grain obligor, échelle 0-100
OBLIGOR_COL = "ID_CLIENT"          # identifiant obligor (None si déjà au grain obligor)
YEAR_COL    = "year"
SEG_COL     = "Seg"
OUT_COL     = "HLC_test_heterogeneite"   # classes de sortie (post fusion hétérogénéité)

## 1. Contrôle des données d'entrée

> `df_brut` doit être chargé en amont (notebook précédent ou cellule de chargement).

In [ ]:
print("Shape           :", df_brut.shape)
print("Classes uniques :", df_brut[CLASS_COL].n_unique())
print("Années          :", sorted(df_brut[YEAR_COL].unique().to_list()))
print("Segments        :", sorted(df_brut[SEG_COL].unique().to_list()))

# vérification des colonnes attendues
for c in (CLASS_COL, LGD_COL, OBLIGOR_COL, YEAR_COL, SEG_COL):
    assert c in df_brut.columns, f"Colonne manquante : {c}"

## 2. Construction du comparateur et calcul des OVL

On instancie le `FusionComparator` avec le seul test OVL (KDE) et on lance `run()`.
Cette étape calcule l'OVL de Weitzman sur **chaque paire adjacente** (classes consécutives triées par LGD moyenne croissante).

Le passage au grain obligor est automatique via `obligor_col` (agrégation `group_by([class, obligor]).mean(lgd)`).

> Durée : ~30 secondes.

In [ ]:
cmp = FusionComparator(
    df_brut,
    class_col=CLASS_COL,
    lgd_col=LGD_COL,
    obligor_col=OBLIGOR_COL,
    pairs="adjacent",
    tests=[OVLTest(method="kde")],
).run()

print(f"{len(cmp.order)} classes ordonnées par LGD croissante")
print(f"{len(cmp.pair_list)} paires adjacentes à évaluer")

## 3. Calibration empirique du seuil OVL

### 3.1 Principe

Plutôt que de fixer un seuil OVL arbitraire (pourquoi 0.50 et pas 0.60 ?), on le calibre sur les données par **stabilité du partitionnement**.

**Méthode** (Lange et al. 2004) : pour chaque seuil candidat, on simule `n_bootstrap` rééchantillonnages des obligors par classe (bootstrap stratifié), on refait la fusion transitive complète à chaque réplicat, puis on mesure la **reproductibilité** du partitionnement obtenu via l'Adjusted Rand Index (ARI, Hubert & Arabie 1985).

- ARI proche de 1 → les réplicats donnent le même découpage → seuil **stable**
- ARI faible → découpage erratique → seuil **instable**

Le seuil retenu **maximise l'ARI** parmi les partitions non-triviales (on exclut les cas dégénérés « tout fusionné » / « rien fusionné » où l'ARI sature artificiellement à 1 — borne de validité de la métrique, pas un hyperparamètre).

> Durée : ~3-5 minutes.

In [ ]:
cmp.calibrate(
    seuils=[round(s, 2) for s in np.arange(0.50, 0.76, 0.05)],
    n_bootstrap=200,
    n_subsample=300,
    method="kde",
    random_state=42,   # fixé pour la reproductibilité du dossier validateur
)

print(cmp.df_calibration)
print(f"\nSeuil OVL retenu (max ARI, partitions non-triviales) : {cmp.seuil_calibre}")

### 3.2 Courbe de stabilité

Visualisation pour le dossier validateur : on veut voir un **pic net** de l'ARI dans la zone non-dégénérée. Les croix rouges marquent les partitions dégénérées (exclues de la sélection).

In [ ]:
df_cal = cmp.df_calibration
s    = df_cal["seuil_ovl"].to_numpy()
ari  = df_cal["ari_moyen"].to_numpy()
std  = df_cal["ari_std"].to_numpy()
n_cl = df_cal["n_classes_median"].to_numpy()
n_initial = len(cmp.order)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.errorbar(s, ari, yerr=std, fmt="o-", color="black", lw=2,
             capsize=3, label="ARI moyen ± écart-type")
ax1.fill_between(s, ari - std, ari + std, alpha=0.12, color="black")

# marquer les partitions dégénérées
for xi, ni, ai in zip(s, n_cl, ari):
    if ni <= 1 or ni >= n_initial:
        ax1.scatter(xi, ai, marker="x", color="red", s=80, zorder=5)

ax1.axvline(cmp.seuil_calibre, color="green", ls=":", lw=2,
            label=f"seuil retenu = {cmp.seuil_calibre}")
ax1.set_xlabel("seuil OVL")
ax1.set_ylabel("ARI (stabilité du partitionnement)")
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=9)
ax1.set_title(
    "Calibration du seuil OVL par stabilité bootstrap\n"
    "(× = partition dégénérée, exclue de la sélection)"
)

ax2 = ax1.twinx()
ax2.plot(s, n_cl, "d--", color="grey", alpha=0.45)
ax2.set_ylabel("n classes finales (médiane)", color="grey")

plt.tight_layout()
plt.show()

## 4. Fusion transitive avec le seuil calibré

`solve()` utilise automatiquement `cmp.seuil_calibre` (fixé par `calibrate()`) pour décider les fusions, puis les enchaîne de manière transitive (si A~B et B~C alors A∪B∪C).

- `label_style="glouton"` reproduit le format historique des labels (`A_16_B_15_C_5`)
- le `log` documente chaque fusion itération par itération

In [ ]:
sol = cmp.solve(criterion="ovl", label_style="glouton")

print(f"{sol['n_initial']} classes → {sol['n_final']} classes finales")
print(f"({sol['n_initial'] - sol['n_final']} fusions effectives)\n")
print(sol["log"])

In [ ]:
# composition détaillée des classes finales
print("Composition des classes finales :")
for classe, composantes in sorted(sol["composition"].items()):
    print(f"  {classe}  <-  {composantes}")

## 5. Diagnostic documentaire (batterie complète)

On relance le comparateur avec **tous** les tests pour documenter le dossier validateur.
L'OVL est paramétré avec le seuil calibré ; les autres tests (Welch/KS/ANOVA/Cohen/Hedges/Wasserstein) sont **documentaires** — ils servent à montrer la convergence (ou divergence) des critères, pas à décider.

Rappel : Welch/KS/ANOVA saturent à grand n (p≈0 partout) → ils ne suggèrent aucune fusion. C'est attendu et documenté comme tel.

In [ ]:
cmp_doc = FusionComparator(
    df_brut,
    class_col=CLASS_COL,
    lgd_col=LGD_COL,
    obligor_col=OBLIGOR_COL,
    pairs="adjacent",
    tests=[
        OVLTest(
            method="kde",
            ovl_modere=cmp.seuil_calibre,
            ovl_strict=min(cmp.seuil_calibre + 0.25, 1.0),
        ),
    ] + [t for t in _default_suite() if not isinstance(t, OVLTest)],
).run()

### 5.1 Vue large : une ligne par paire

Colonnes : OVL, verdict de chaque test (oui/non), `n_pour_fusion` (nb de tests favorables), `accord_total` (tous les tests d'accord).

In [ ]:
with pl.Config(tbl_cols=-1, tbl_rows=-1):
    print(cmp_doc.matrix())

### 5.2 Vue détaillée : une ligne par (paire, test)

Le grain le plus fin — statistic et p_value de chaque test pour chaque paire.

In [ ]:
with pl.Config(tbl_rows=30):
    print(cmp_doc.detail())

### 5.3 Consensus par test

Par test : nombre de **paires** fusionnables et taux d'accord avec l'OVL.

⚠️ `n_paires_fusionnables` = nombre de paires adjacentes où le test dit oui — **pas** le nombre de classes finales (qui dépend de la fusion transitive, voir 5.4).

In [ ]:
print(cmp_doc.consensus())

### 5.4 Consensus étendu : classes finales par test

Lance `solve()` pour chaque test et donne le **nombre de classes finales** (après fusion transitive) + leur composition. C'est ici qu'on voit le vrai impact de chaque critère sur la granularité du modèle.

In [ ]:
with pl.Config(fmt_str_lengths=200, tbl_rows=-1):
    print(cmp_doc.consensus_avec_solve(label_style="glouton"))

### 5.5 Heatmap des décisions

Carte visuelle paires × tests : vert = fusion suggérée, rouge = non, gris = n/a.

In [ ]:
cmp_doc.plot_heatmap()
plt.show()

### 5.6 Distributions LGD par paire

Grille des densités superposées pour chaque paire adjacente. L'aire verte = recouvrement (OVL). Le titre indique la valeur OVL et le verdict ✓/✗ par rapport au seuil calibré.

In [ ]:
cmp.plot_all_pairs(ncols=3)
plt.show()

### 5.7 Exploration d'une paire spécifique

Tableau de bord complet (distributions, OVL, ECDF, tailles d'effet, boxplot, récap) pour une paire donnée. Ici la première fusion du log.

In [ ]:
if sol["log"].height > 0:
    CLASSE_A = sol["log"]["groupe_a"][0]
    CLASSE_B = sol["log"]["groupe_b"][0]

    a = df_brut.filter(pl.col(CLASS_COL) == CLASSE_A).get_column(LGD_COL).to_numpy()
    b = df_brut.filter(pl.col(CLASS_COL) == CLASSE_B).get_column(LGD_COL).to_numpy()

    suite = HeterogeneityTestSuite(a, b, label_a=CLASSE_A, label_b=CLASSE_B).fit_all()
    print(suite.summary())
    suite.plot_all()
    plt.show()
else:
    print("Aucune fusion réalisée — rien à explorer.")

## 6. Application à la base

On rattache la colonne de classe finale `HLC_test_heterogeneite` à la base, via le mapping retenu (critère OVL, seuil calibré). Les classes absentes du mapping (écartées car n<2) gardent leur valeur d'origine.

In [ ]:
df_final = df_brut.with_columns(
    pl.col(CLASS_COL)
    .cast(pl.Utf8)
    .replace(sol["mapping"])
    .alias(OUT_COL)
)

# contrôle : effectif obligor et LGD moyenne par classe finale
recap = (
    df_final
    .group_by(OUT_COL)
    .agg(
        pl.col(OBLIGOR_COL).n_unique().alias("n_obligors"),
        pl.col(LGD_COL).mean().round(2).alias("lgd_moyenne"),
    )
    .sort("lgd_moyenne")
)
print(recap)

## 7. Récapitulatif pour le dossier validateur

In [ ]:
print("=" * 64)
print("TESTS D'HÉTÉROGÉNÉITÉ — RÉCAPITULATIF")
print("=" * 64)
print(f"Classes initiales (CHR_apres_fusion)   : {sol['n_initial']}")
print(f"Classes finales   (HLC_test_hetero)    : {sol['n_final']}")
print(f"Fusions effectives                     : {sol['n_initial'] - sol['n_final']}")
print()
print(f"Critère de fusion : OVL Weitzman (1970), estimation KDE")
print(f"Seuil retenu      : {cmp.seuil_calibre}")
print(f"Calibration       : stabilité bootstrap (ARI, Hubert & Arabie 1985)")
print(f"                    200 réplicats, sous-échantillon 300 obligors/classe")
print(f"                    sélection : max ARI sur partitions non-triviales")
print()
print("Journal de fusion :")
with pl.Config(tbl_rows=-1):
    print(sol["log"].select(["iteration", "groupe_a", "groupe_b", "gap_lgd", "score"]))
print("=" * 64)

## 8. Exports Excel (optionnel)

Pour le dossier validateur : tableau de calibration ARI, matrice des tests, journal de fusion.

In [ ]:
cmp.df_calibration.write_excel("calibration_seuil_ovl.xlsx")
cmp_doc.matrix().write_excel("tests_heterogeneite_matrix.xlsx")
sol["log"].write_excel("journal_fusion_heterogeneite.xlsx")

print("Exports réalisés :")
print("  - calibration_seuil_ovl.xlsx")
print("  - tests_heterogeneite_matrix.xlsx")
print("  - journal_fusion_heterogeneite.xlsx")